# Week 4, Lab 4 — Multi-Agent Workflow with LangGraph (Local Models)

**Course:** Agentic AI Engineering — Local Models Edition
**Author:** Abhishek

Builds a small LangGraph graph with a **Router** node that inspects the
question and dispatches it to either a **Researcher** node (knowledge
lookup) or a **Calculator** node (arithmetic) — then routes back through
a final **Responder** node that writes the answer.

Runs on Colab (Hugging Face) or your local PC (Ollama) — same graph code
either way.


## 1. Environment detection & install

In [3]:
BACKEND = "ollama"


## 2. Build the `llm` object (same interface either way)

In [4]:
if BACKEND == "huggingface":
    from transformers import pipeline
    from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

    pipe = pipeline(
        "text-generation",
        model="Qwen/Qwen2.5-1.5B-Instruct",
        max_new_tokens=200,
    )
    llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

else:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="qwen2.5:3b", temperature=0.2)

print("llm ready:", llm)


llm ready: metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}} model='qwen2.5:3b' temperature=0.2


## 3. Define the graph state and the tools each node can use

In [5]:
from typing import TypedDict, Literal
import ast, operator as op

class GraphState(TypedDict):
    question: str
    route: str
    tool_result: str
    answer: str

# --- Calculator tool (safe eval, no raw eval()) ---
_ALLOWED_OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
                ast.Div: op.truediv, ast.Pow: op.pow, ast.USub: op.neg}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("unsupported expression")

def calculator(expression: str) -> str:
    try:
        return str(_safe_eval(ast.parse(expression, mode="eval").body))
    except Exception as e:
        return f"Error: {e}"

# --- Mock knowledge base (stand-in for a real retriever from Week 1 / RAG module) ---
_MOCK_KB = {
    "langgraph": "LangGraph is a library for building stateful, multi-step LLM "
                 "workflows as graphs of nodes and edges.",
    "ollama": "Ollama runs open-weight LLMs locally and exposes an OpenAI-compatible API.",
    "mcp": "The Model Context Protocol standardizes how agents connect to external tools "
           "and data sources.",
}

def lookup_fact(topic: str) -> str:
    return _MOCK_KB.get(topic.lower().strip(), f"No local knowledge entry for '{topic}'.")

print("Tools ready: calculator, lookup_fact")


Tools ready: calculator, lookup_fact


## 4. Define the graph nodes

- `router_node` — asks the LLM to classify the question as `"math"` or `"research"`
- `researcher_node` — pulls a topic out of the question and calls `lookup_fact`
- `calculator_node` — pulls an expression out of the question and calls `calculator`
- `responder_node` — writes the final natural-language answer from the tool result


In [6]:
def router_node(state: GraphState) -> GraphState:
    classification_prompt = (
        "Classify this question as exactly one word, 'math' or 'research'. "
        f"Question: {state['question']}\nAnswer with just one word."
    )
    result = llm.invoke(classification_prompt).content.strip().lower()
    route = "math" if "math" in result else "research"
    return {**state, "route": route}


def calculator_node(state: GraphState) -> GraphState:
    extract_prompt = (
        "Extract only the arithmetic expression from this question, "
        f"with no words: {state['question']}"
    )
    expression = llm.invoke(extract_prompt).content.strip()
    result = calculator(expression)
    return {**state, "tool_result": f"Calculator({expression}) = {result}"}


def researcher_node(state: GraphState) -> GraphState:
    extract_prompt = (
        "Extract only the single topic keyword being asked about "
        f"(no full sentence): {state['question']}"
    )
    topic = llm.invoke(extract_prompt).content.strip()
    result = lookup_fact(topic)
    return {**state, "tool_result": f"Lookup({topic}) = {result}"}


def responder_node(state: GraphState) -> GraphState:
    prompt = (
        f"Question: {state['question']}\n"
        f"Tool result: {state['tool_result']}\n"
        "Write a short, direct final answer for the user based on the tool result."
    )
    answer = llm.invoke(prompt).content.strip()
    return {**state, "answer": answer}


def route_decision(state: GraphState) -> Literal["math", "research"]:
    return state["route"]


## 5. Wire the graph together

In [7]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(GraphState)
graph.add_node("router", router_node)
graph.add_node("calculator", calculator_node)
graph.add_node("researcher", researcher_node)
graph.add_node("responder", responder_node)

graph.add_edge(START, "router")
graph.add_conditional_edges("router", route_decision, {
    "math": "calculator",
    "research": "researcher",
})
graph.add_edge("calculator", "responder")
graph.add_edge("researcher", "responder")
graph.add_edge("responder", END)

app = graph.compile()
print("Graph compiled.")


Graph compiled.


## 6. Run it

In [8]:
result = app.invoke({
    "question": "What is 45 * 12 + 30?",
    "route": "", "tool_result": "", "answer": "",
})
print("Route taken:", result["route"])
print("Tool result:", result["tool_result"])
print("Final answer:", result["answer"])


Route taken: math
Tool result: Calculator(45 * 12 + 30) = 570
Final answer: The result of 45 * 12 + 30 is 570.


In [ ]:
result = app.invoke({
    "question": "What is LangGraph?",
    "route": "", "tool_result": "", "answer": "",
})
print("Route taken:", result["route"])
print("Tool result:", result["tool_result"])
print("Final answer:", result["answer"])


## 7. Exercise for students

1. Add a third route (`"chitchat"`) for questions that need neither tool —
   route straight to `responder` with no tool call.
2. Replace `lookup_fact` with a real retriever from your Week 1/RAG
   material — embed 5–10 short documents and retrieve the closest match.
3. Visualize the graph: `app.get_graph().print_ascii()` (or `.draw_mermaid()`
   if you want a Mermaid diagram to drop into your README).
4. **Compare frameworks:** rebuild this exact router+2-tools flow using
   CrewAI (Week 3) or the OpenAI Agents SDK (Week 2) with the same local
   model. Which framework's code do you find easiest to reason about?

## Where this goes next
Week 5 (`5_agent_frameworks`) does the same router+tools pattern in
AutoGen and Pydantic AI — good for direct side-by-side comparison.
